# DengAI-UNT: Protocolo Experimental F6-F10 con Optuna y GPU
## Comparación Rigurosa de 5 Arquitecturas Espacio-Temporales sobre Datos Epidemiológicos de La Libertad

Este notebook ejecuta el protocolo experimental formal exigido por el **Anexo A de la Ficha Técnica SP-01**:
- **Modelos Evaluados**:
  1. Individual 1: XGBoost (Tabular / Gradient Boosting)
  2. Individual 2: LSTM (Recurrente Temporal)
  3. Individual 3: ConvLSTM (Espacio-Temporal 2D)
  4. Híbrido 1: CNN-LSTM (Convolucional + Recurrente)
  5. Híbrido 2: ConvLSTM-Attention (Mecanismo de atención temporal y espacial)
- **Optimización Bayesiana**:
  - 100 trials por modelo con Optuna (TPE Sampler + Median Pruner).
  - Optimización **exclusivamente sobre Train y Validación interna** (el conjunto de prueba final jamás participa).
- **Estrategia de Validación**:
  - Walk-forward temporal (26 semanas de validación, 26 semanas de prueba final).
  - Escaladores ajustados únicamente con datos de entrenamiento (cero fuga de datos).
- **Pruebas Estadísticas Obligatorias (Anexo A)**:
  - Prueba de Friedman sobre **bloques independientes** (semanas epidemiológicas de prueba).
  - Comparaciones pareadas post-hoc de Wilcoxon con corrección múltiple de Holm.
  - Prueba de concordancia/discordancia de McNemar sobre clasificación de brote.
  - Prueba pareada de DeLong sobre curvas ROC pareadas.
- **Empaquetado Automático**:
  - Al finalizar, exporta los modelos entrenados (.joblib), tablas CSV/JSON y figuras en 300 DPI y los descarga automáticamente en un archivo ZIP a tu PC.

In [ ]:
# 1. Verificación del entorno y aceleración por GPU
import torch
print(f"PyTorch version: {torch.__version__}")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Dispositivo activo: {device}")
if device == 'cuda':
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")
    print(f"VRAM disponible: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("Aviso: Ejecutando en CPU. Si estás en Google Colab, activa GPU en: Entorno de ejecución -> Cambiar tipo de entorno de ejecución -> T4 GPU.")

In [ ]:
# 2. Clonación o sincronización del repositorio DengAI-UNT e instalación compatible
import os
import subprocess

repo_url = "https://github.com/JosephBanGH/DengAI-UNT.git"
repo_dir = "/content/DengAI-UNT" if os.path.isdir("/content") else os.path.abspath("DengAI-UNT")

if os.path.basename(os.getcwd()) == "DengAI-UNT" and os.path.isdir("backend"):
    repo_dir = os.getcwd()
elif not os.path.isdir(os.path.join(repo_dir, "backend")):
    print("Clonando repositorio DengAI-UNT...")
    subprocess.run(["git", "clone", repo_url, repo_dir], check=True)

os.chdir(repo_dir)
subprocess.run(["git", "pull", "origin", "main"], check=True)
print(f"Directorio de trabajo activo: {os.getcwd()}")

# Instalación preservando el entorno PyTorch CUDA nativo de Colab
!pip install -q optuna pydantic-settings xgboost shap httpx bcrypt PyJWT

In [ ]:
# 3. Importación de módulos del pipeline DengAI e ingesta 100% REAL (MINSA + NASA POWER)
import os
import sys

if os.path.isdir("/content/DengAI-UNT/backend"):
    os.chdir("/content/DengAI-UNT")
backend_dir = os.path.abspath("backend")
if backend_dir not in sys.path:
    sys.path.insert(0, backend_dir)

from sqlalchemy import select, func
from app.db.base import Base
from app.db.session import engine, SessionLocal
from app.db.seed import ensure_geo_units, ensure_data_sources
from app.services.real_data_service import ingest_real_data
from app.models.epi import DengueCase, ClimateObservation
from app.ml.features import load_panel, build_dataset
from app.ml.models.base import ModelData
from app.ml.models.registry import create_model, SEARCH_SPACES
from app.ml.metrics import evaluate_all
from app.ml.stats_tests import compare_models, mcnemar_test, delong_roc_test

# Asegurar esquema de base de datos y catálogos base
Base.metadata.create_all(bind=engine)
db = SessionLocal()
ensure_geo_units(db)
ensure_data_sources(db)
db.commit()

real_cases = db.scalar(select(func.count()).select_from(DengueCase).where(DengueCase.is_synthetic.is_(False))) or 0
real_climate = db.scalar(select(func.count()).select_from(ClimateObservation).where(ClimateObservation.is_synthetic.is_(False))) or 0

if real_cases == 0 or real_climate == 0:
    print("Descargando e ingiriendo datos oficiales 100% REALES (MINSA/CDC Perú + NASA POWER 2015-2024)...")
    ingest_res = ingest_real_data(db, start_year=2015, end_year=2024, actor="colab_optuna")
    print(f"Ingesta completada -> Modo efectivo: {ingest_res['effective_data_mode']}")
else:
    print(f"Datos reales ya presentes en SQLite: {real_cases} registros MINSA y {real_climate} registros NASA POWER.")

# Exigir explícitamente mode='real' (prohibido caer silenciosamente en datos sintéticos)
panel = load_panel(db, mode='real')
assert not panel.is_synthetic and panel.data_mode == 'real', "ERROR CRÍTICO: El panel cargado no es 100% real."
print(f"Panel REAL verificado: {len(panel.panel)} registros, {len(panel.districts)} distritos, {len(panel.weeks)} semanas ({panel.weeks[0]} a {panel.weeks[-1]})")
print(f"Modo: {panel.data_mode} | Es sintético: {panel.is_synthetic}")

# Construir dataset walk-forward sin fuga temporal (H=1 semana)
bundle = build_dataset(panel, horizon=1, test_weeks=26, validation_weeks=26)
print(f"Particiones: Train={len(bundle.y_train)}, Val={len(bundle.y_val)}, Test={len(bundle.y_test)}")

train_data = ModelData(bundle.X_train, bundle.seq_train, bundle.grid_train, bundle.cell_train, bundle.y_train, bundle.feature_names)
val_data = ModelData(bundle.X_val, bundle.seq_val, bundle.grid_val, bundle.cell_val, bundle.y_val, bundle.feature_names)
test_data = ModelData(bundle.X_test, bundle.seq_test, bundle.grid_test, bundle.cell_test, bundle.y_test, bundle.feature_names)

In [ ]:
# 4. Optimización Bayesiana con Optuna (100 trials por modelo con Median Pruner)
import optuna
import numpy as np

optuna.logging.set_verbosity(optuna.logging.WARNING)

models = ['xgboost', 'lstm', 'convlstm', 'cnn_lstm', 'convlstm_attention']
studies = {}
best_params_all = {}
N_TRIALS = 100

for algo in models:
    print(f"\n========================================================")
    print(f">>> Optimizando hiperparámetros: {algo.upper()} ({N_TRIALS} trials) <<<")
    print(f"========================================================")
    space = SEARCH_SPACES[algo]
    
    def objective(trial):
        params = {}
        for name, spec in space.items():
            if spec['type'] == 'int':
                params[name] = trial.suggest_int(name, spec['low'], spec['high'], step=spec.get('step', 1))
            elif spec['type'] == 'float':
                params[name] = trial.suggest_float(name, spec['low'], spec['high'], log=spec.get('log', False))
            elif spec['type'] == 'categorical':
                params[name] = trial.suggest_categorical(name, spec['choices'])
        
        model = create_model(algo)
        # Se pasa trial=trial para activar la poda temprana (MedianPruner)
        model.fit(train_data, val_data, params=params, trial=trial, epochs=25)
        val_pred = model.predict(val_data)
        val_mae = float(np.mean(np.abs(bundle.y_val - val_pred)))
        return val_mae
    
    study = optuna.create_study(
        direction='minimize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=5),
    )
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    studies[algo] = study
    best_params_all[algo] = study.best_params
    complete_cnt = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
    pruned_cnt = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f"Mejor Val MAE ({algo}): {study.best_value:.4f}")
    print(f"Trials completados: {complete_cnt} | Trials podados precozmente: {pruned_cnt}")

In [ ]:
# 5. Entrenamiento Final con Mejores Hiperparámetros y Evaluación en Test (Sin Fuga)
import pandas as pd

test_predictions = {}
final_metrics = {}
trained_models = {}
outbreak_threshold = float(np.quantile(bundle.y_train, 0.90))

for algo in models:
    print(f"Entrenando modelo final {algo.upper()} con mejores hiperparámetros...")
    best_params = best_params_all[algo]
    model = create_model(algo)
    model.fit(train_data, val_data, params=best_params, epochs=40)
    y_test_pred = np.clip(model.predict(test_data), 0, None)
    test_predictions[algo] = y_test_pred
    trained_models[algo] = model
    
    m = evaluate_all(
        bundle.y_test,
        y_test_pred,
        y_insample=bundle.y_train[-100:],
        outbreak_threshold=outbreak_threshold,
    )
    final_metrics[algo] = m

df_metrics = pd.DataFrame(final_metrics).T
print(f"\n--- RENDIMIENTO EN CONJUNTO DE PRUEBA (Umbral de brote = {outbreak_threshold:.2f} casos) ---")
display(df_metrics[['MAE', 'RMSE', 'MASE', 'AUC_PR', 'sensitivity', 'specificity', 'Brier']])

In [ ]:
# 6. Pruebas Estadísticas Rigurosas (Anexo A de la Ficha Técnica SP-01)
test_blocks = bundle.meta_test['target_week'].astype(str).tolist()
stats_result = compare_models(test_predictions, bundle.y_test, blocks=test_blocks)

print("=" * 70)
print("1. PRUEBA DE FRIEDMAN SOBRE BLOQUES INDEPENDIENTES (SEMANAS)")
print("=" * 70)
print(f"Bloques evaluados: {stats_result['blocks']} semanas independientes")
print(f"Estadístico Chi-cuadrado: {stats_result['friedman']['statistic']:.4f}")
print(f"P-valor: {stats_result['friedman']['p_value']:.4e}")
print(f"Diferencia global significativa (p < 0.05): {stats_result['friedman']['significant']}")
print("Rangos medios (menor es mejor):", stats_result['friedman']['mean_ranks'])

print("\n" + "=" * 70)
print("2. COMPARACIONES PAREADAS DE WILCOXON CON CORRECCIÓN DE HOLM")
print("=" * 70)
df_pairwise = pd.DataFrame(stats_result['pairwise'])
cols_wilcoxon = ['comparison', 'statistic', 'p_value', 'p_value_adjusted', 'significant_holm', 'median_loss_diff']
display(df_pairwise[cols_wilcoxon])

# 3. McNemar pareado sobre clasificación de brote
print("\n" + "=" * 70)
print(f"3. PRUEBA DE MCNEMAR (PARES DISCORDANTES EN CLASIFICACIÓN DE BROTE >= {outbreak_threshold:.2f})")
print("=" * 70)
binary_actual = (bundle.y_test >= outbreak_threshold).astype(int)
binary_preds = {algo: (test_predictions[algo] >= outbreak_threshold).astype(int) for algo in models}

mcnemar_rows = []
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        m1, m2 = models[i], models[j]
        res = mcnemar_test(binary_actual, binary_preds[m1], binary_preds[m2], m1, m2)
        if res.get('applicable'):
            mcnemar_rows.append({
                'comparacion': f"{m1} vs {m2}",
                'p_value': res['p_value'],
                'significativo': res['significant'],
                'discordantes_b': res['b'],
                'discordantes_c': res['c'],
                'interpretacion': res['interpretation'],
            })
df_mcnemar = pd.DataFrame(mcnemar_rows)
display(df_mcnemar)

# 4. DeLong pareado sobre curvas ROC
print("\n" + "=" * 70)
print("4. PRUEBA DE DELONG (DIFERENCIA ENTRE CURVAS ROC PAREADAS)")
print("=" * 70)
delong_rows = []
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        m1, m2 = models[i], models[j]
        res = delong_roc_test(binary_actual, test_predictions[m1], test_predictions[m2], m1, m2)
        if res.get('applicable'):
            delong_rows.append({
                'comparacion': f"{m1} vs {m2}",
                'auc_1': res['auc_a'],
                'auc_2': res['auc_b'],
                'z_stat': res['statistic'],
                'p_value': res['p_value'],
                'significativo': res['significant'],
            })
df_delong = pd.DataFrame(delong_rows)
display(df_delong)

In [ ]:
# 7. Generación de Gráficos de Publicación (Formato IMRyD para Manuscrito/Tesis)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Boxplot de errores absolutos por modelo
errors_dict = {algo.upper(): np.abs(bundle.y_test - test_predictions[algo]) for algo in models}
df_err = pd.DataFrame(errors_dict)
sns.boxplot(data=df_err, ax=axes[0], palette='Set2')
axes[0].set_title('Distribución del Error Absoluto en Conjunto de Prueba', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Error Absoluto (Casos por Distrito-Semana)')
axes[0].set_xlabel('Modelo')

# 2. Curvas de Convergencia Bayesiana (Optuna)
for algo, st in studies.items():
    completed_trials = [t for t in st.trials if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None]
    if completed_trials:
        vals = [t.value for t in completed_trials]
        best_so_far = np.minimum.accumulate(vals)
        axes[1].plot(best_so_far, label=algo.upper(), linewidth=2)

axes[1].set_title('Convergencia de Optimización Bayesiana (Optuna TPE)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Trial Completado')
axes[1].set_ylabel('Mejor MAE de Validación')
axes[1].legend()

plt.tight_layout()
plt.savefig('comparacion_modelos_optuna.png', dpi=300, bbox_inches='tight')
print("Figura 1 guardada: comparacion_modelos_optuna.png (300 DPI)")
plt.show()

# Gráfico adicional: Curvas ROC Superpuestas
plt.figure(figsize=(7, 6))
for algo in models:
    fpr, tpr, _ = roc_curve(binary_actual, test_predictions[algo])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{algo.upper()} (AUC = {roc_auc:.3f})", linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Línea base (Azar)')
plt.title('Curvas ROC Pareadas en Detección de Brotes', fontsize=12, fontweight='bold')
plt.xlabel('Tasa de Falsos Positivos (1 - Especificidad)')
plt.ylabel('Tasa de Verdaderos Positivos (Sensibilidad)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('curvas_roc_modelos.png', dpi=300, bbox_inches='tight')
print("Figura 2 guardada: curvas_roc_modelos.png (300 DPI)")
plt.show()

In [ ]:
# 8. Empaquetado Automático y Descarga Directa al Navegador
import os
import shutil
import json
import joblib

export_dir = "paquete_resultados_dengai"
os.makedirs(f"{export_dir}/modelos", exist_ok=True)
os.makedirs(f"{export_dir}/tablas_manuscrito", exist_ok=True)
os.makedirs(f"{export_dir}/figuras_manuscrito", exist_ok=True)

print("Empaquetando artefactos para publicación...")

# 1. Guardar modelos entrenados con pesos e hiperparámetros óptimos
for algo, model in trained_models.items():
    model_payload = {
        "model": model,
        "algorithm": algo,
        "best_params": best_params_all.get(algo, {}),
        "scaler_mean": bundle.scaler_mean,
        "scaler_std": bundle.scaler_std,
        "feature_names": bundle.feature_names,
    }
    joblib.dump(model_payload, f"{export_dir}/modelos/{algo}_modelo_optimizado.joblib")

# 2. Exportar tablas formateadas para tesis / paper
df_metrics.to_csv(f"{export_dir}/tablas_manuscrito/tabla_1_metricas_rendimiento.csv")
df_pairwise[cols_wilcoxon].to_csv(f"{export_dir}/tablas_manuscrito/tabla_2_wilcoxon_holm.csv", index=False)
if len(df_mcnemar):
    df_mcnemar.to_csv(f"{export_dir}/tablas_manuscrito/tabla_3_mcnemar_brote.csv", index=False)
if len(df_delong):
    df_delong.to_csv(f"{export_dir}/tablas_manuscrito/tabla_4_delong_roc.csv", index=False)

with open(f"{export_dir}/tablas_manuscrito/mejores_hiperparametros_optuna.json", "w", encoding="utf-8") as f:
    json.dump(best_params_all, f, indent=2)

# 3. Exportar figuras en alta resolución
for fig_file in ["comparacion_modelos_optuna.png", "curvas_roc_modelos.png"]:
    if os.path.exists(fig_file):
        shutil.copy(fig_file, f"{export_dir}/figuras_manuscrito/{fig_file}")

# 4. Comprimir todo en un archivo ZIP estructurado
zip_name = "resultados_experimentos_dengai"
zip_path = shutil.make_archive(zip_name, 'zip', export_dir)
print(f"\n>>> PAQUETE GENERADO EXITOSAMENTE: {zip_path} <<<")

# 5. Disparar descarga automática al navegador (en Google Colab)
try:
    from google.colab import files
    print("Descargando paquete ZIP directamente a tu computadora...")
    files.download(f"{zip_name}.zip")
except Exception:
    print(f"El archivo comprimido '{zip_name}.zip' está disponible para descarga manual en la barra lateral de archivos.")